In [127]:
import sys
# adjust path
sys.path.append('../../../NER-german-telegram')

from src.helpers.db_helpers import execute_sql_select
import src.config.db_credentials as db

from datetime import datetime
from random import sample

import spacy
from spacy import displacy

## Data

In [128]:
table_name = "linked_from_twitter"
channel_name = 'QUERDENKEN_711'
ts_before = datetime.now()
query = f"""SELECT * FROM {table_name} WHERE channel_name = '{channel_name}'"""
data = execute_sql_select(command=query, database=db.DB_NAME_TELEGRAM, return_result_as_df=True)
ts_after = datetime.now()

print(f"Took {ts_after - ts_before}")

Column names:  ['channel_name', 'channel_id', 'channel_description', 'message_id', 'from_id', 'via_bot_id', 'date', 'edit_date', 'text', 'forwards', 'fwd_from', 'replies', 'reply_to', 'media', 'views', 'id']
Connection to DB closed
Took 0:00:00.696865


In [129]:
data.head()

,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,date,edit_date,text,forwards,fwd_from,replies,reply_to,media,views,id
0,QUERDENKEN_711,1228827167,"Hier erfahrt ihr, was in unserer Initiative pa...",2149,None,None,2021-03-31 19:07:46,NaT,Thema:\nDAS JAHR DER FREIHEIT UND DES FRIEDENS...,93.0,None,None,None,MessageMediaWebPage(webpage=WebPage(id=5069470...,32380.0,QUERDENKEN_71112288271672149
1,QUERDENKEN_711,1228827167,"Hier erfahrt ihr, was in unserer Initiative pa...",2114,None,None,2021-03-29 02:59:59,2021-03-29 03:00:00,ACHTUNG - Updates während der Demo\nUns haben ...,45.0,None,None,None,MessageMediaWebPage(webpage=WebPageEmpty(id=89...,26842.0,QUERDENKEN_711122882716721142021-03-29 03:00:00
2,QUERDENKEN_711,1228827167,"Hier erfahrt ihr, was in unserer Initiative pa...",379,None,None,2020-07-17 15:52:29,NaT,via QUERDENKEN 711 - Wir für das Grundgesetz h...,297.0,None,None,None,MessageMediaWebPage(webpage=WebPageEmpty(id=15...,22350.0,QUERDENKEN_7111228827167379
3,QUERDENKEN_711,1228827167,"Hier erfahrt ihr, was in unserer Initiative pa...",2440,None,None,2021-04-24 03:07:59,NaT,Ihr seid auf der Suche nach einer Initiative i...,19.0,None,None,None,MessageMediaWebPage(webpage=WebPage(id=4705731...,31407.0,QUERDENKEN_71112288271672440
4,QUERDENKEN_711,1228827167,"Hier erfahrt ihr, was in unserer Initiative pa...",1366,None,None,2021-01-04 08:56:05,2021-01-04 08:56:06,https://youtu.be/ooM3rrBoiBA\n\n@QUERDENKEN_71...,479.0,None,None,None,MessageMediaWebPage(webpage=WebPage(id=5127856...,39225.0,QUERDENKEN_711122882716713662021-01-04 08:56:06


In [149]:
df = data[['id', 'text']]

In [150]:
texts = df.text.sample(5, random_state=2)

## Spacy

In [151]:
nlp = spacy.load("de_core_news_sm")

In [152]:
for text in texts: 
    
    if text: 
        doc = nlp(text)
    
        if doc.ents:
            for ent in doc.ents:
                pass
                # print(ent.text, ent.label_)

In [153]:
for text in texts: 
    
    try: 
        
        doc = nlp(text)
        sentence_spans = list(doc.sents)
        displacy.render(sentence_spans, style="ent")
        
    except:
        pass

/Users/elisabeth/repos/NER-german-telegram/venv/lib/python3.9/site-packages/spacy/displacy/__init__.py:205: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


## Flair (default)

In [160]:
from flair.data import Sentence
from flair.models import SequenceTagger

# load tagger
tagger_default = SequenceTagger.load("flair/ner-german")

2022-05-23 15:13:43,816 loading file /Users/elisabeth/.flair/models/ner-german/a125be40445295f7e94d0afdb742cc9ac40ec4e93259dc30f35220ffad9bf1f6.f46c4c5cfa5e34baa838983373e30051cd1cf1e933499408a49e451e784b0a11
2022-05-23 15:14:01,704 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>


In [162]:
tagger_lg = SequenceTagger.load("flair/ner-german-large")

Downloading:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

2022-05-23 15:16:13,091 loading file /Users/elisabeth/.flair/models/ner-german-large/6b8de9edd73722050be2547acf64c037b2df833c6e8f0e88934de08385e26c1e.4b0797effcc6ebb1889d5d29784b97f0a099c1569b319d87d7c387e44e2bba48


Downloading:   0%|          | 0.00/616 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/4.83M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

2022-05-23 15:17:05,842 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>


In [166]:
def get_ents(tagger, text):

    sentence = Sentence(text)

    # predict NER tags
    tagger.predict(sentence)

    # print sentence
    print(sentence)

    # print predicted NER spans
    print('The following NER tags are found:')
    # iterate over entities and print
    for entity in sentence.get_spans('ner'):
        print(entity)

In [167]:
for t in texts:
    
    try:
        get_ents(tagger_default, t)
    except:
        pass

Sentence: "Nikolai Nerling # Volkslehrer # Holocaust - Auf QUERDENKEN-Demonstrationen nicht willkommen via https :// www.youtube.com / watch ? v = luY7oIHQUnM @ QUERDENKEN _ 711 - Diskussion & Austausch @ QUERDENKEN711 - Bilder und Videos von Demos aus 711 / STUTTGART hochladen @ QUERDENKEN711 _ aktiv" → ["Nikolai Nerling"/PER, "Holocaust"/ORG, "watch"/ORG, "QUERDENKEN"/ORG, "STUTTGART"/LOC, "QUERDENKEN711"/ORG]
The following NER tags are found:
Span[0:2]: "Nikolai Nerling" → PER (0.9987)
Span[5:6]: "Holocaust" → ORG (0.5738)
Span[16:17]: "watch" → ORG (0.6038)
Span[22:23]: "QUERDENKEN" → ORG (0.665)
Span[40:41]: "STUTTGART" → LOC (0.9967)
Span[43:44]: "QUERDENKEN711" → ORG (0.6507)
Sentence: "DEMO-TICKER Tag der Freiheit oder der Diktatur ? Berlin verbietet „ Querdenken “- Großdemo und elf weitere Versammlungen Nun ist es amtlich : Die Berliner Behörden haben die Querdenken-Veranstaltung vom 1. August verboten . Ein Eilantrag auf " einstweiligen Rechtsschutz " ist beim Verwaltungsgeri

In [168]:
for t in texts:
    
    try:
        get_ents(tagger_lg, t)
    except:
        pass

Sentence: "Nikolai Nerling # Volkslehrer # Holocaust - Auf QUERDENKEN-Demonstrationen nicht willkommen via https :// www.youtube.com / watch ? v = luY7oIHQUnM @ QUERDENKEN _ 711 - Diskussion & Austausch @ QUERDENKEN711 - Bilder und Videos von Demos aus 711 / STUTTGART hochladen @ QUERDENKEN711 _ aktiv" → ["Nikolai Nerling"/PER, "Holocaust"/MISC, "luY7oIHQUnM"/MISC, "QUERDENKEN"/LOC, "QUERDENKEN711"/ORG, "STUTTGART"/LOC, "QUERDENKEN711"/ORG]
The following NER tags are found:
Span[0:2]: "Nikolai Nerling" → PER (1.0)
Span[5:6]: "Holocaust" → MISC (0.9999)
Span[20:21]: "luY7oIHQUnM" → MISC (0.9542)
Span[22:23]: "QUERDENKEN" → LOC (1.0)
Span[30:31]: "QUERDENKEN711" → ORG (0.7743)
Span[40:41]: "STUTTGART" → LOC (1.0)
Span[43:44]: "QUERDENKEN711" → ORG (0.987)
Sentence: "DEMO-TICKER Tag der Freiheit oder der Diktatur ? Berlin verbietet „ Querdenken “- Großdemo und elf weitere Versammlungen Nun ist es amtlich : Die Berliner Behörden haben die Querdenken-Veranstaltung vom 1. August verboten . E